# Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve,average_precision_score, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import ast
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
DATA_DIR = BASE_DIR / "data"

# load the input data

## load OneHotEncoding as y

In [ ]:
OHE_train = pd.read_csv(DATA_DIR / "OHE_train_dataset_All_Food.csv", sep=';', header=0, index_col=0)
OHE_test = pd.read_csv(DATA_DIR / "OHE_test_dataset_All_Food.csv", sep=';', header=0, index_col=0)

y_train = np.array([
    np.array(ast.literal_eval(f), dtype=int)
    for f in OHE_train['OneHotEncoding']
])
y_test = np.array([
    np.array(ast.literal_eval(f), dtype=int)
    for f in OHE_test['OneHotEncoding']
])

## load Mordred as x

In [ ]:
X_train_index = pd.read_csv(DATA_DIR / 'X_train_features_Mordred.csv', sep=',')
X_test_index = pd.read_csv(DATA_DIR / 'X_test_features_Mordred.csv', sep=',')
X_test = np.delete(X_test_index, 0, 1)
X_train = np.delete(X_train_index, 0, 1)

# ML models

## optimized RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = "RF"
number_e = 1100
min_samples_l = 3
min_samples_s = 2

RFCLassifier = RandomForestClassifier(
    random_state=42,
    n_estimators=number_e,
    min_samples_leaf=min_samples_l,
    min_samples_split=min_samples_s
)
RFCLassifier.fit(X_train, y_train)
y_scores_RF = RFCLassifier.predict_proba(X_test)
y_pred_RF = RFCLassifier.predict(X_test)

## optimized CatBoost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.multiclass import OneVsRestClassifier

classifier = "Cat"
learning = 0.01
dep = 6

Cat = OneVsRestClassifier(CatBoostClassifier(
    n_estimators=1000,
    depth=dep,
    learning_rate=learning,
    loss_function='Logloss',
    random_seed=42,
    verbose=False
))
Cat.fit(X_train, y_train)
y_scores_Cat = Cat.predict_proba(X_test)
y_pred_Cat = Cat.predict(X_test)

# ROC and PR-curve

In [ ]:
lists = pd.read_csv(DATA_DIR / 'Total_dataset_All_Food.csv', sep=';', header=0)["Labels"].apply(ast.literal_eval)
unique_items = list(set(item for sublist in lists for item in sublist))

OHE = pd.DataFrame(
    OHE_test["OneHotEncoding"].apply(ast.literal_eval).tolist(),
    index=OHE_test["SMILES"],
    columns=unique_items
)
OHE_num = OHE.to_numpy().flatten()

model_preds = {
    "RandomForest": np.transpose(np.array(RFCLassifier.predict_proba(X_test))[:, :, 1]).flatten(),
    "CatBoost": Cat.predict_proba(X_test).flatten(),
}
labels = list(model_preds.keys())

fpr_RF, tpr_RF, roc_auc_RF, avg_precision_RF_micro, PR_auc_RF, precision_RF, recall_RF = []

plt.figure(figsize=(6, 5))
for i, (label, results) in enumerate(model_preds.items()):
    fprRFC, tprRFC, _ = roc_curve(OHE_num, results)
    fpr_RF.append(fprRFC)
    tpr_RF.append(tprRFC)
    roc_auc_RF.append(auc(fprRFC, tprRFC))

    precisionRFC, recallRFC, _ = precision_recall_curve(OHE_num, results)
    precision_RF.append(precisionRFC)
    recall_RF.append(recallRFC)

    avg_precision_RF_micro.append(average_precision_score(OHE_num, results, average="micro"))
    PR_auc_RF.append(auc(recall_RF[i], precision_RF[i]))

    print(f'{label} ROC AUC:       {roc_auc_RF[i]:.4f}')
    print(f'{label} PR AUC:        {PR_auc_RF[i]:.4f}')
    print(f'{label} avg_precision: {avg_precision_RF_micro[i]:.4f}')

    plt.plot(fpr_RF[i], tpr_RF[i], label=f'{label} (AUC = {roc_auc_RF[i]:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.grid()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
for i, label in enumerate(labels):
    plt.plot(recall_RF[i], precision_RF[i], label=f'{label} (AUC = {PR_auc_RF[i]:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR Curve')
plt.legend(loc="upper right")
plt.grid()
plt.tight_layout()
plt.show()


# Classification metrics summary

In [ ]:
model_bin_preds = {
    "RandomForest": y_pred_RF,
    "CatBoost": y_pred_Cat,
}
average_type = "macro"
print(f"{'Model':<15} {'F1':<14} {'Avg Score (AP)':<18} {'Precision':<20} {'Recall'}")
print("-" * 82)

metrics_rows = []
for label, y_pred in model_bin_preds.items():
    y_score_flat = model_preds[label]

    f1 = f1_score(y_test.flatten(), y_pred.flatten(), average=average_type, zero_division=0)
    avg_score = average_precision_score(OHE_num, y_score_flat, average=average_type)
    precision = precision_score(y_test.flatten(), y_pred.flatten(), average=average_type, zero_division=0)
    recall = recall_score(y_test.flatten(), y_pred.flatten(), average=average_type, zero_division=0)

    metrics_rows.append({
        "Model": label,
        f"F1 ({average_type})": round(f1, 4),
        "Avg Score (AP)":  round(avg_score, 4),
        f"Precision ({average_type})": round(precision, 4),
        f"Recall ({average_type})": round(recall, 4),
    })
    print(f"{label:<15} {f1:<14.4f} {avg_score:<18.4f} {precision:<20.4f} {recall:.4f}")

metrics_df = pd.DataFrame(metrics_rows).set_index("Model")
display(metrics_df.style
    .format("{:.4f}")
    .set_caption(f"Classification Metrics — {average_type}-averaged")
    .set_table_styles([{"selector": "caption","props": [("font-size", "14px"), ("font-weight", "bold")]}])
    .background_gradient(cmap="YlGn", axis=0)
)
